In [8]:
from openai import OpenAI
from neo4j import GraphDatabase
import json

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "sternenkrieger"))

print("Setup ✅")

Setup ✅


In [9]:
driver

In [10]:
def get_personen() -> str:
    with driver.session() as session:
        result = session.run("""
            MATCH (p:Person)
            RETURN p.name as name, p.rolle as rolle, p.status as status
        """)
        return str([dict(r) for r in result])

def get_netzwerk(name: str) -> str:
    with driver.session() as session:
        result = session.run("""
            MATCH (p:Person {name: $name})-[:KENNT]->(b:Person)
            RETURN b.name as name, b.rolle as rolle
        """, name=name)
        return str([dict(r) for r in result])

def get_ueberweisungen() -> str:
    with driver.session() as session:
        result = session.run("""
            MATCH (a:Person)-[r:ÜBERWIES]->(b:Person)
            RETURN a.name as von, b.name as an, r.betrag as betrag
        """)
        return str([dict(r) for r in result])

# Tool-Registry — Agent schaut hier nach
TOOLS = {
    "get_personen": get_personen,
    "get_netzwerk": get_netzwerk,
    "get_ueberweisungen": get_ueberweisungen,
}

print(f"Tools verfügbar: {list(TOOLS.keys())} ✅")

Tools verfügbar: ['get_personen', 'get_netzwerk', 'get_ueberweisungen'] ✅


In [19]:
import re

SYSTEM_PROMPT = """Du bist ein Ermittlungsassistent mit Zugriff auf folgende Tools:

- get_personen() → alle Personen mit Rolle und Status
- get_netzwerk(name) → alle bekannten Personen einer Person  
- get_ueberweisungen() → alle Geldüberweisungen

Um ein Tool aufzurufen, schreibe EXAKT so:
TOOL: tool_name
ARGS: {"parameter": "wert"}

Wenn du keine Args brauchst:
TOOL: get_personen
ARGS: {}

Wenn du die Antwort kennst, schreibe:
ANTWORT: deine Antwort hier

Denke Schritt für Schritt. Rufe Tools auf bis du genug weißt."""

In [20]:
def parse_tool_call(text: str):
    """Erkennt ob LLM ein Tool aufrufen will"""
    tool_match = re.search(r'TOOL:\s*(\w+)', text)
    args_match = re.search(r'ARGS:\s*({.*?})', text, re.DOTALL)
    answer_match = re.search(r'ANTWORT:\s*(.+)', text, re.DOTALL)
    
    if answer_match:
        return "answer", answer_match.group(1).strip(), {}
    if tool_match:
        tool_name = tool_match.group(1).strip()
        args = json.loads(args_match.group(1)) if args_match else {}
        return "tool", tool_name, args
    return "unknown", text, {}


In [27]:
def run_agent(frage: str, max_steps: int = 18):
    print(f"\n🔍 Frage: {frage}")
    print("─" * 50)
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": frage}
    ]
    
    for step in range(max_steps):
        response = client.chat.completions.create(
            model="qwen2.5:14b",
            messages=messages,
            temperature=0
        )
        
        llm_output = response.choices[0].message.content
        messages.append({"role": "assistant", "content": llm_output})
        
        action_type, value, args = parse_tool_call(llm_output)
        
        if action_type == "answer":
            print(f"🕵️  {value}")
            return value
            
        elif action_type == "tool":
            print(f"🔧 Tool: {value}({args})")
            
            if value in TOOLS:
                tool_func = TOOLS[value]
                result = tool_func(**args) if args else tool_func()
                print(f"📊 Result: {result[:1000]}...")
                
                # Ergebnis zurück ans LLM
                messages.append({
                    "role": "user", 
                    "content": f"Tool-Ergebnis: {result}\nWas ist deine Schlussfolgerung?"
                })
            else:
                messages.append({
                    "role": "user",
                    "content": f"Tool '{value}' existiert nicht. Verfügbar: {list(TOOLS.keys())}"
                })

        else:
            print(f"🤔 LLM verliert Faden, nudge...")  
            messages.append({
            "role": "user",
            "content": "Bitte antworte jetzt mit ANTWORT: basierend auf dem was du bereits weißt. Stelle keine Fragen."
        })
            
    print("⚠️ Max Steps erreicht")

    print(50*"-")
    print(messages)


print("Agent bereit ✅")

Agent bereit ✅


In [28]:
run_agent("Wie viel Geld hat Walter White insgesamt überwiesen?")


🔍 Frage: Wie viel Geld hat Walter White insgesamt überwiesen?
──────────────────────────────────────────────────
🔧 Tool: get_ueberweisungen({})
📊 Result: [{'von': 'Walter White', 'an': 'Dagobert Duck', 'betrag': 8400}, {'von': 'Walter White', 'an': 'Darth Vader', 'betrag': 3500}, {'von': 'Dagobert Duck', 'an': 'Auslandskontakt', 'betrag': 12000}]...
🕵️  Walter White hat insgesamt 11900 überwiesen.


'Walter White hat insgesamt 11900 überwiesen.'